# Run Josh's experiment with attention masking

Choose an **L4 or A100 GPU**, then **Run all**. This launcher downloads the masking branch and executes the existing `colab.ipynb` runner with the preset below. It loads the archived run, runs the checks and smoke pass, then completes Part 0, H1, H2 and the new H1 masking experiment. Previously saved answers are skipped.

The default is Josh's original Qwen3-4B/bf16 sample: seed 0 and 100 questions per cell. The masking stage uses two-fact toy and two-hop MuSiQue questions that pass Part 0, under AA/AB/BA/BB. On the archived sample that is 740 question rows: 740 each for blocked/gold/except-gold/full and 640 for the distractor control. The smoke uses an unfiltered sample; any extra smoke rows outside the composition subset are excluded from the full tables.

For the full benchmark across three layouts, change `SEEDS` to `[0, 1, 2]` and `N_PER_CELL` to `None`. Part 0 will complete the additional probes before masking starts. That also expands the original H1/H2 runs.

The source defaults to the published fork branch so Josh can run before merging the pull request. After merging, `REPO` can be changed to `https://github.com/JoshBong/latent-dispatch.git` and `COMMIT` to `main`. The exact resolved commit is printed and saved with results. A fresh runtime (or a new `ROOT`) avoids reusing a checkout from an older experiment.

Download `/content/results-Qwen3-4B-bfloat16.tar.gz` when finished. It contains previous and new rows, masking manifests, and executed experiment notebooks. Keep the browser tab open while running.

In [ ]:
REPO = "https://github.com/ajit-mallav/latent-dispatch.git"
COMMIT = "masking-experiment"
ROOT = "/content/latent-dispatch-masking"
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SEEDS = [0]
N_PER_CELL = 100
LOAD = ["Qwen3-4B-bfloat16-2026-11-13"]
NOTEBOOKS = ["part0_ruler", "h1_encoding", "h2_thinking", "h1_masking"]
SKIP = {"h2_thinking": ["t1_1024"]}  # same omitted text-report control as Josh's original run
RUN_TESTS = True
GATE_BOXED, GATE_CUT = 50.0, 20.0

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "Select an L4 or A100 GPU runtime before running this notebook"
if DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support bf16; select an L4 or A100")
checkout = Path(ROOT)
if not checkout.exists():
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    subprocess.run(["git", "-C", ROOT, "checkout", COMMIT], check=True)
else:
    origin = subprocess.check_output(["git", "-C", ROOT, "remote", "get-url", "origin"], text=True).strip()
    actual = subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip()
    wanted = subprocess.check_output(["git", "-C", ROOT, "rev-parse", f"{COMMIT}^{{commit}}"], text=True).strip()
    if origin != REPO or actual != wanted:
        raise RuntimeError("ROOT contains another source checkout; use a fresh runtime or choose a new ROOT")
if not (checkout / "notebooks/h1_masking.ipynb").exists():
    raise RuntimeError("This checkout predates masking; use a fresh runtime or choose a new ROOT")
os.chdir(ROOT)
subprocess.run(["git", "log", "-1", "--format=running commit %h (%s)"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "papermill>=2.6"], check=True)

In [ ]:
import papermill as pm

pm.execute_notebook(
    "colab.ipynb", "/content/masking-runner.ipynb", kernel_name="python3", log_output=True,
    parameters=dict(REPO=REPO, COMMIT=COMMIT, ROOT=ROOT, MODEL=MODEL, DTYPE=DTYPE,
                    SEEDS=SEEDS, N_PER_CELL=N_PER_CELL, LOAD=LOAD, NOTEBOOKS=NOTEBOOKS,
                    SKIP=SKIP, RUN_TESTS=RUN_TESTS, GATE_BOXED=GATE_BOXED, GATE_CUT=GATE_CUT))

The common runner prints per-arm coverage, output-quality audits, paired EM and gold-answer likelihood comparisons. Download the results tarball from Colab's Files panel before deleting the runtime. A stopped run resumes on the same runtime from saved question rows.